## Experimental Workflow

The purpose of this section is to provide an overview of the typical experimental process.  

The experiment typically follows these steps:

1. **Data Overview**  
   Inspect the dataset and understand its basic features.

2. **Preprocessing**  
   Clean the data and handle missing or invalid values if necessary.

3. **Dataset Splitting**  
   Split the data into training, validation (if required), and test sets.

4. **Data Normalisation**  
   Apply appropriate normalisation or standardisation to the data.

5. **Model Definition and Parameter Settings**  
   Define the BNN architecture and specify model and inference parameters.

6. **Training (and Validation, if applicable)**  
   Train the model and monitor performance during training.

7. **Evaluation and Analysis**  
   Obtain results, evaluate model performance, and analyse the outcomes.


In [7]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

### Data Overview

- Dataset size
- Features
- Target variables

For the Iris dataset, details can be found in the corresponding API documentation.


https://scikit-learn.org/stable/datasets/toy_dataset.html#iris-dataset

In [12]:
iris = load_iris()
X, y = iris.data, iris.target
print(X.shape, y.shape)

(150, 4) (150,)


### Preprocessing

- Check for missing values.
- More complex datasets may require more advanced cleaning procedures.

In [15]:
# combine features and target into one DataFrame
df = pd.DataFrame(X, columns=iris.feature_names)
df["target"] = y

df.columns = ["sepal_len", "sepal_wid", "petal_len", "petal_wid", "target"]
print(df.head())

# check missing values
print(df.isnull().sum())

   sepal_len  sepal_wid  petal_len  petal_wid  target
0        5.1        3.5        1.4        0.2       0
1        4.9        3.0        1.4        0.2       0
2        4.7        3.2        1.3        0.2       0
3        4.6        3.1        1.5        0.2       0
4        5.0        3.6        1.4        0.2       0
sepal_len    0
sepal_wid    0
petal_len    0
petal_wid    0
target       0
dtype: int64


### Dataset Splitting

- Split the dataset using methods provided by `scikit-learn`.
- Common train–test splits are 80:20 or 70:30.
- A validation set may be required when tuning hyperparameters or monitoring model performance during training.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

### Data Normalisation

- Required when feature scales differ significantly.
- Not necessary for categorical features.
- For classification tasks, categorical variables may require one-hot encoding.
- This step is optional and depends on the dataset and model.

In [ ]:
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)
#Normalisation parameters should be learned from the training set and applied to the test set.

#covert numpy array to tensors
X_tr = torch.tensor(X_tr, dtype=torch.float32)
y_tr = torch.tensor(y_tr, dtype=torch.long)

X_te = torch.tensor(X_te, dtype=torch.float32)
y_te = torch.tensor(y_te, dtype=torch.long)

### Data Loader for Training

- Wrap the training data into a `TensorDataset`.
- Use a `DataLoader` to enable mini-batch training.
- Set `batch_size` and enable shuffling for training.

In [3]:
train_dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=16, shuffle=True)

Test accuracy: 0.8999999761581421


### Model Definition and Parameter Settings

- Define the model architecture.
- Choose an optimizer.
- Specify the loss function.

In [ ]:
# 2) built-in model
model = torch.nn.Linear(4, 3)   # 4 features, 3 classes
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = torch.nn.CrossEntropyLoss()

### Training

- Set the model to training mode.
- Clear gradients from the previous iteration (`zero_grad`).
- Compute the loss for the current batch.
- Perform backpropagation to compute gradients.
- Update model parameters using the optimizer.

In [ ]:
model.train() #Set the model to training mode.
for _ in range(100):
    for xb, yb in train_dl:
        opt.zero_grad()
        #Clear gradients from the previous iteration
        loss = loss_fn(model(xb), yb)
        #Compute the loss for the current batch.
        loss.backward()
        #Perform backpropagation to compute gradients.
        opt.step()
        #Update model parameters using the optimizer.

### Evaluation and Analysis

- Set the model to evaluation mode (`model.eval()`).
- Disable gradient computation during evaluation.
- Obtain predictions on the test set.
- Compute evaluation metrics.
- Analyse the results.

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(X_te).argmax(dim=1)
    acc = (pred == y_te).float().mean()

print("Test accuracy:", acc.item())